# Ingestion Pipeline

## Importing Required Modules

In [35]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from dotenv import load_dotenv
from langchain_groq import ChatGroq

## Document

In [8]:
def pdf_loader():
    folder_path = "data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs+=1
    print(f"no. of docs = {num_docs}")
    print(f"no. of pages = {len(all_docs)}")
    return all_docs

In [14]:
documents = pdf_loader()

no. of docs = 0
no. of pages = 0


## Chunks

In [13]:
def chunkify(document, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
    chunked_docs = text_splitter.split_documents(document)
    return chunked_docs


In [15]:
chunked_docs = chunkify(documents)

## Embedding

In [18]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print(embeddings.shape)
        return embeddings


In [19]:
embedding_manager = EmbeddingManager()

loading model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions =  384


C:\Users\muska\AppData\Local\Temp\ipykernel_4612\617255078.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())


## Vector Store

In [22]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embedding for RAG"}
        )

        print("Initialised the vector store for pdf documents")
        print("collection count=", self.collection.count())

    def add_to_store(self, documents, embeddings):
        if len(documents)!= len(embeddings):
            raise ValueError("length of document and embeddings doesn't match")

        ids = []
        all_metadata = []
        document_content = []
        embeddings_list = []

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):

            ids.append(f"doc_{uuid.uuid4()}")
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)

            document_content.append(doc.page_content)

            embeddings_list.append(embeddings.tolist())

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=all_metadata,
                documents=document_content
            )

        print("total documents added in the vector store:", len(document_content))
        print("docs in collection:", self.collection.count())




In [23]:
vector_store_manager = VectorStoreManager()

Initialised the vector store for pdf documents
collection count= 0


In [24]:
texts = [doc.page_content for doc in chunked_docs]
embeddings = embedding_manager.generate_embeddings(texts)
vector_store = vector_store_manager.add_to_store(chunked_docs, embeddings)


Batches: 0it [00:00, ?it/s]

(0,)
total documents added in the vector store: 0
docs in collection: 0


# Retrieval Pipeline

In [31]:
class RAGRetriever :
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        query_embedding = embedding_manager.generate_embeddings(query[0])

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results = top_k
        )

        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            documents = results["documents"][0]

            for i, (id,metadata,distance,document) in enumerate(zip(ids,metadatas,distances,documents)):
                similarity_score = 1-distance

                if similarity_score>=score_threshold:
                    retrieved_docs.append({
                        "id":id,
                        "metadata":metadata,
                        "document":document,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i+1        
                    })

            print(f"retrieved {len(retrieved_docs)} documents")
            
        else:
            print("no documents found")
            
        return retrieved_docs



In [33]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [34]:
rag_retriever.retrieve("What is Rag?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(384,)


AttributeError: 'NoneType' object has no attribute 'collection'

# Integrating LLM

In [38]:
load_dotenv()
api_key = os.getenv("Groq_API_KEY")

In [39]:
llm = ChatGroq(
    api_key=api_key,
    model="openai/gpt-oss-120b",
    max_tokens=1024,
    temperature=0.1
)

In [40]:
def generate_output(query, retriever, llm, top_k=5):
    results = retriever.retrieve(query,top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We couldn't find relevent response to the query")

    prompt = f"""Use the given context to generate the answer for the query
                    context:{context}
                    query:{query}"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [41]:
answer = generate_output("What is Rag?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(384,)


AttributeError: 'NoneType' object has no attribute 'collection'

In [42]:
print(answer)

NameError: name 'answer' is not defined